In [0]:
import math

CATALOG   = "e-commerce"
SILVER_DB = "silver"
GOLD_DB   = "gold"

In [0]:
spark.sql(f"USE CATALOG `{CATALOG}`")
spark.sql(f"CREATE DATABASE IF NOT EXISTS {GOLD_DB}")

spark.table(f"{SILVER_DB}.fact_sales").createOrReplaceTempView("fact_sales")
spark.table(f"{SILVER_DB}.dim_products").createOrReplaceTempView("dim_products")
spark.table(f"{SILVER_DB}.dim_calendar").createOrReplaceTempView("dim_calendar")
print("Gold DB ready | Silver views registered")

In [0]:
df_product_revenue = spark.sql("""
    SELECT
        p.product_sk,
        p.product_id,
        p.product_name,
        p.brand,
        p.category,
        p.cocoa_tier,
        p.weight_band,
        p.cocoa_percent,
        p.weight_g,

        COUNT(f.order_id)                               AS total_orders,
        SUM(f.quantity)                                 AS total_units_sold,
        ROUND(SUM(f.gross_sales),   2)                  AS gross_sales,
        ROUND(SUM(f.discount_amount), 2)                AS total_discount,
        ROUND(SUM(f.revenue),       2)                  AS total_revenue,
        ROUND(SUM(f.cost),          2)                  AS total_cost,
        ROUND(SUM(f.profit),        2)                  AS total_profit,
        ROUND(AVG(f.profit_margin) * 100, 2)            AS avg_profit_margin_pct,
        ROUND(SUM(f.revenue) / COUNT(f.order_id), 2)   AS aov,

        RANK() OVER (ORDER BY SUM(f.revenue) DESC)      AS revenue_rank,
        RANK() OVER (ORDER BY SUM(f.profit)  DESC)      AS profit_rank,
        RANK() OVER (ORDER BY SUM(f.quantity) DESC)     AS units_rank

    FROM fact_sales   f
    JOIN dim_products p ON f.product_sk = p.product_sk
    WHERE f.is_valid = 1
    GROUP BY
        p.product_sk, p.product_id, p.product_name,
        p.brand, p.category, p.cocoa_tier,
        p.weight_band, p.cocoa_percent, p.weight_g
    ORDER BY total_revenue DESC
""")

df_product_revenue.show(20, truncate=False)

row_count = df_product_revenue.count()
num_partitions = max(2, math.ceil(row_count / 10000))

df_product_revenue.repartition(num_partitions).write \
    .format("delta").mode("overwrite") \
    .option("overwriteSchema", "true") \
    .saveAsTable(f"{GOLD_DB}.product_revenue_ranking")

print(f"gold.product_revenue_ranking written - {row_count} products ({num_partitions} partitions)")

In [0]:
df_top10 = spark.sql("""
    SELECT
        p.product_name,
        p.brand,
        p.category,
        p.cocoa_tier,

        SUM(f.quantity)                                 AS total_units_sold,
        COUNT(f.order_id)                               AS total_orders,
        ROUND(SUM(f.revenue), 2)                        AS total_revenue,
        ROUND(SUM(f.profit),  2)                        AS total_profit,
        ROUND(AVG(f.profit_margin)*100, 2)              AS avg_margin_pct,

        RANK() OVER (ORDER BY SUM(f.quantity) DESC)     AS units_rank,
        RANK() OVER (ORDER BY SUM(f.revenue)  DESC)     AS revenue_rank

    FROM fact_sales   f
    JOIN dim_products p ON f.product_sk = p.product_sk
    WHERE f.is_valid = 1
    GROUP BY p.product_name, p.brand, p.category, p.cocoa_tier
    ORDER BY total_units_sold DESC
    LIMIT 10
""")

print("TOP 10 BEST-SELLING PRODUCTS (by units)")
df_top10.show(truncate=False)

df_top10.repartition(2).write \
    .format("delta").mode("overwrite") \
    .option("overwriteSchema", "true") \
    .saveAsTable(f"{GOLD_DB}.top10_products")

print(f"gold.top10_products written")

In [0]:
df_category = spark.sql("""
    SELECT
        p.category,
        p.cocoa_tier,

        COUNT(DISTINCT p.product_sk)                    AS product_count,
        COUNT(f.order_id)                               AS total_orders,
        SUM(f.quantity)                                 AS total_units_sold,
        ROUND(SUM(f.revenue),  2)                       AS total_revenue,
        ROUND(SUM(f.profit),   2)                       AS total_profit,
        ROUND(AVG(f.profit_margin)*100, 2)              AS avg_margin_pct,
        ROUND(SUM(f.revenue) / COUNT(f.order_id), 2)   AS aov,

        ROUND(
            SUM(f.revenue) * 100.0
            / SUM(SUM(f.revenue)) OVER (), 2
        )                                               AS revenue_share_pct,

        RANK() OVER (ORDER BY SUM(f.revenue) DESC)      AS revenue_rank

    FROM fact_sales   f
    JOIN dim_products p ON f.product_sk = p.product_sk
    WHERE f.is_valid = 1
    GROUP BY p.category, p.cocoa_tier
    ORDER BY total_revenue DESC
""")

df_category.show(truncate=False)

row_count = df_category.count()
num_partitions = max(2, math.ceil(row_count / 10000))

df_category.repartition(num_partitions).write \
    .format("delta").mode("overwrite") \
    .option("overwriteSchema", "true") \
    .saveAsTable(f"{GOLD_DB}.category_performance")

print(f"gold.category_performance written - {row_count} rows")

In [0]:
df_margin_issue = spark.sql("""
    WITH product_metrics AS (
        SELECT
            p.product_sk,
            p.product_name,
            p.brand,
            p.category,
            ROUND(SUM(f.revenue), 2)            AS total_revenue,
            ROUND(SUM(f.profit),  2)             AS total_profit,
            ROUND(AVG(f.profit_margin)*100, 2)   AS avg_margin_pct,
            COUNT(f.order_id)                    AS total_orders,

            PERCENT_RANK() OVER (ORDER BY SUM(f.revenue) DESC)        AS revenue_pct_rank,
            PERCENT_RANK() OVER (ORDER BY AVG(f.profit_margin) ASC)   AS margin_pct_rank

        FROM fact_sales   f
        JOIN dim_products p ON f.product_sk = p.product_sk
        WHERE f.is_valid = 1
        GROUP BY p.product_sk, p.product_name, p.brand, p.category
    )
    SELECT
        product_name,
        brand,
        category,
        total_revenue,
        total_profit,
        avg_margin_pct,
        total_orders,

        CASE
            WHEN avg_margin_pct < 10 THEN 'Critical - margin < 10%'
            WHEN avg_margin_pct < 20 THEN 'Warning - margin < 20%'
            WHEN avg_margin_pct < 30 THEN 'Watch - margin < 30%'
            ELSE 'Healthy'
        END AS margin_status,

        CASE
            WHEN revenue_pct_rank <= 0.25 AND margin_pct_rank <= 0.25
            THEN 'HIGH REVENUE / LOW MARGIN'
            ELSE 'Normal'
        END AS alert_flag

    FROM product_metrics
    ORDER BY avg_margin_pct ASC
""")

print("MARGIN ISSUE PRODUCTS (lowest margin first)")
df_margin_issue.show(20, truncate=False)

print("HIGH REVENUE / LOW MARGIN (problem products)")
df_margin_issue.filter(df_margin_issue.alert_flag == "HIGH REVENUE / LOW MARGIN").show(truncate=False)

row_count = df_margin_issue.count()
num_partitions = max(2, math.ceil(row_count / 10000))

df_margin_issue.repartition(num_partitions).write \
    .format("delta").mode("overwrite") \
    .option("overwriteSchema", "true") \
    .saveAsTable(f"{GOLD_DB}.margin_issue_products")

print(f"gold.margin_issue_products written - {row_count} rows")

In [0]:
df_cocoa = spark.sql("""
    SELECT
        p.cocoa_tier,
        p.cocoa_percent,

        COUNT(DISTINCT p.product_sk)                    AS products_in_tier,
        COUNT(f.order_id)                               AS total_orders,
        SUM(f.quantity)                                 AS total_units_sold,
        ROUND(SUM(f.revenue),  2)                       AS total_revenue,
        ROUND(SUM(f.profit),   2)                       AS total_profit,
        ROUND(AVG(f.profit_margin)*100, 2)              AS avg_margin_pct,
        ROUND(AVG(f.unit_price), 2)                     AS avg_unit_price,
        ROUND(SUM(f.revenue) / COUNT(f.order_id), 2)   AS aov,

        ROUND(
            SUM(f.revenue) * 100.0
            / SUM(SUM(f.revenue)) OVER (), 2
        )                                               AS revenue_share_pct

    FROM fact_sales   f
    JOIN dim_products p ON f.product_sk = p.product_sk
    WHERE f.is_valid = 1
    GROUP BY p.cocoa_tier, p.cocoa_percent
    ORDER BY p.cocoa_percent DESC
""")

print("COCOA % IMPACT ON SALES")
df_cocoa.show(30, truncate=False)

print("SUMMARY BY COCOA TIER")
spark.sql("""
    SELECT
        p.cocoa_tier,
        COUNT(DISTINCT p.product_sk)    AS products,
        ROUND(SUM(f.revenue), 2)        AS total_revenue,
        ROUND(AVG(f.profit_margin)*100,2) AS avg_margin_pct,
        ROUND(AVG(f.unit_price), 2)     AS avg_price
    FROM fact_sales   f
    JOIN dim_products p ON f.product_sk = p.product_sk
    WHERE f.is_valid = 1
    GROUP BY p.cocoa_tier
    ORDER BY total_revenue DESC
""").show(truncate=False)

row_count = df_cocoa.count()
num_partitions = max(2, math.ceil(row_count / 10000))

df_cocoa.repartition(num_partitions).write \
    .format("delta").mode("overwrite") \
    .option("overwriteSchema", "true") \
    .saveAsTable(f"{GOLD_DB}.cocoa_sales_analysis")

print(f"gold.cocoa_sales_analysis written - {row_count} rows")

In [0]:
print("GOLD - Product Analysis COMPLETE")
print("  gold.product_revenue_ranking  - Revenue/profit by product")
print("  gold.top10_products           - Top 10 best-sellers")
print("  gold.category_performance     - Category performance")
print("  gold.margin_issue_products    - High rev / low margin")
print("  gold.cocoa_sales_analysis     - Cocoa % vs sales")
print()

for tbl in ["product_revenue_ranking", "top10_products", "category_performance", "margin_issue_products", "cocoa_sales_analysis"]:
    cnt = spark.table(f"{GOLD_DB}.{tbl}").count()
    print(f"   gold.{tbl:<30} - {cnt:,} rows")